# 34. Computer Use & Browser Agents

**Tier:** Frontier
**Estimated time:** 45 minutes
**Prerequisites:** 19, 28
**Priority:** 🟢 Nice-to-have — a major modality, but niche unless your product automates GUIs; the concepts (screenshot→action loop) build directly on Tier 4 and can be picked up when needed. *If skipped, revisit when:* a task requires automating software without an API, or a role mentions browser/RPA-style agents.
**Source material:** Anthropic's computer-use tool documentation; Stanford Lecture 7 (agentic LLMs)

## What You'll Learn
- The screenshot → action loop that every computer-use / browser agent runs underneath
- The Anthropic computer-use tool schema, and how it differs from a normal function-call tool
- A real, working demo: an agent that looks at an image of a mock calculator and clicks buttons to compute an answer
- Why accessibility-tree/selector-based browser agents are usually more robust than raw-pixel agents

## Why This Matters
Most tools an agent uses (notebook 19) have a clean API. Computer use and browser agents exist for the much larger space of software that DOESN'T — legacy internal tools, third-party sites with no API, desktop apps. The tradeoff is real: pixel-level control is powerful but brittle (a UI redesign breaks every hardcoded coordinate), which is exactly why this notebook is marked nice-to-have rather than crucial — reach for an API-based tool (notebook 19/28) whenever one exists, and reserve this for when it genuinely doesn't.


In [ ]:
import os, io, base64
from PIL import Image, ImageDraw

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

HAS_ANTHROPIC = bool(os.environ.get("ANTHROPIC_API_KEY"))
TEACH_MODEL = "claude-haiku-4-5-20251001"   # this teach model supports vision + tool use

if HAS_ANTHROPIC:
    import anthropic
    client = anthropic.Anthropic()
    print("Anthropic ready.")
else:
    client = None
    print("No ANTHROPIC_API_KEY — the live vision-agent cells will be skipped.")


## The screenshot → action loop

A computer-use agent runs a variant of notebook 19's tool loop with one twist: instead of calling a `web_search`-style function, its tool returns a *screenshot* (an image), and its action is a *coordinate* (where to click, what to type) rather than a named function argument. The loop shape is otherwise identical: send the model what it can currently see, let it decide an action, execute the action, take a new screenshot, repeat until done.

## The computer-use tool schema

Anthropic's computer-use tools (`computer`, `text_editor`, `bash`) are special-cased tool types the model has been specifically trained to use well — you don't write their JSON schema by hand the way you would for `web_search`; you declare the tool type and the model already knows its action vocabulary (`screenshot`, `left_click`, `type`, `key`, `scroll`, etc.), scoped to a declared screen resolution:

```python
tools = [{
    "type": "computer_20250124",
    "name": "computer",
    "display_width_px": 1024,
    "display_height_px": 768,
}]
```

Using the REAL version of this tool requires the `computer-use` beta header and an actual display to screenshot and control (typically a VM or a container running a virtual display) — infrastructure this notebook doesn't stand up. Instead, we build a small mock environment and use plain vision + a custom `click` tool, which exercises the exact same screenshot → reason → click → repeat loop without needing a real OS-level display.

## Building a mock GUI: a tiny calculator

We render a calculator as an actual image (not a text description) using PIL, with known pixel bounding boxes per button — the agent only ever sees the rendered image, exactly as a real computer-use agent only sees a screenshot, never the underlying button coordinates we know as the environment's author.

In [ ]:
BUTTON_LAYOUT = [
    ["7", "8", "9"],
    ["4", "5", "6"],
    ["1", "2", "3"],
    ["0", "+", "="],
]
BTN_W, BTN_H, MARGIN, DISPLAY_H = 160, 120, 20, 90
IMG_W = MARGIN * 2 + BTN_W * 3
IMG_H = DISPLAY_H + MARGIN * 2 + BTN_H * 4

def button_bbox(row, col):
    x0 = MARGIN + col * BTN_W
    y0 = DISPLAY_H + MARGIN + row * BTN_H
    return x0, y0, x0 + BTN_W - 8, y0 + BTN_H - 8

try:
    from PIL import ImageFont
    _FONT = ImageFont.truetype("/System/Library/Fonts/Supplemental/Arial.ttf", 36)
    _DISPLAY_FONT = ImageFont.truetype("/System/Library/Fonts/Supplemental/Arial.ttf", 32)
except Exception:
    _FONT = _DISPLAY_FONT = ImageFont.load_default()

def render_calculator(display_text):
    img = Image.new("RGB", (IMG_W, IMG_H), "white")
    draw = ImageDraw.Draw(img)
    draw.rectangle([0, 0, IMG_W, DISPLAY_H], outline="black", width=3)
    draw.text((MARGIN, DISPLAY_H // 2 - 16), display_text, fill="black", font=_DISPLAY_FONT)
    for r, row in enumerate(BUTTON_LAYOUT):
        for c, label in enumerate(row):
            x0, y0, x1, y1 = button_bbox(r, c)
            draw.rectangle([x0, y0, x1, y1], outline="black", width=3, fill="#f0f0f0")
            text_bbox = draw.textbbox((0, 0), label, font=_FONT)
            tw, th = text_bbox[2] - text_bbox[0], text_bbox[3] - text_bbox[1]
            draw.text(((x0 + x1) // 2 - tw // 2, (y0 + y1) // 2 - th // 2 - text_bbox[1]),
                      label, fill="black", font=_FONT)
    return img

def image_to_b64(img):
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return base64.standard_b64encode(buf.getvalue()).decode()

demo_img = render_calculator("0")
demo_img.save("/tmp/nb34_calc_demo.png")
print(f"Rendered a {IMG_W}x{IMG_H}px calculator image with {sum(len(r) for r in BUTTON_LAYOUT)} buttons.")


## A `click` tool over pixel coordinates

The agent's ONLY action is `click(x, y)` — exactly the action vocabulary a real computer-use tool exposes, just implemented by us instead of Anthropic's trained-in handler. We map the clicked coordinate to whichever button's bounding box contains it, update calculator state, and re-render — the new screenshot is what the agent sees next.

In [ ]:
CLICK_TOOL_SCHEMA = {
    "name": "click",
    "description": "Click at a pixel coordinate in the current screenshot.",
    "input_schema": {
        "type": "object",
        "properties": {"x": {"type": "integer"}, "y": {"type": "integer"}},
        "required": ["x", "y"],
    },
}

class CalculatorState:
    def __init__(self):
        self.expression = ""
        self.display = "0"

    def click(self, x, y):
        for r, row in enumerate(BUTTON_LAYOUT):
            for c, label in enumerate(row):
                x0, y0, x1, y1 = button_bbox(r, c)
                if x0 <= x <= x1 and y0 <= y <= y1:
                    if label == "=":
                        try:
                            self.display = str(eval(self.expression, {"__builtins__": {}}))
                        except Exception:
                            self.display = "Error"
                        self.expression = ""
                    else:
                        self.expression += label
                        self.display = self.expression
                    return label
        return None   # clicked outside any button

def button_center(row, col):
    x0, y0, x1, y1 = button_bbox(row, col)
    return (x0 + x1) // 2, (y0 + y1) // 2

state = CalculatorState()
cx, cy = button_center(2, 1)   # row 2, col 1 = the "2" button
clicked = state.click(cx, cy)
print(f"Clicked button: {clicked!r}, display now: {state.display!r}")


## Running the agent: look, click, repeat

The agent receives the CURRENT screenshot as an image, decides a click via the `click` tool, we execute it against `CalculatorState`, render the new screenshot, and feed that back — the same message-append loop as notebook 19, with an image in place of a text tool-result.

In [ ]:
def run_computer_use_agent(goal, max_steps=8):
    if not HAS_ANTHROPIC:
        return None, "[skipped: no ANTHROPIC_API_KEY]"
    state = CalculatorState()
    screenshot = render_calculator(state.display)
    messages = [{
        "role": "user",
        "content": [
            {"type": "text", "text": goal},
            {"type": "image", "source": {"type": "base64", "media_type": "image/png",
                                          "data": image_to_b64(screenshot)}},
        ],
    }]
    for step in range(max_steps):
        resp = client.messages.create(model=TEACH_MODEL, max_tokens=300,
                                       system="You control a calculator by clicking pixel coordinates in "
                                              "the screenshot. Click digits/operators, then '=' to compute.",
                                       tools=[CLICK_TOOL_SCHEMA], messages=messages)
        if resp.stop_reason != "tool_use":
            final = " ".join(b.text for b in resp.content if b.type == "text")
            return state.display, final
        messages.append({"role": "assistant", "content": resp.content})
        results = []
        for b in resp.content:
            if b.type == "tool_use":
                clicked_label = state.click(b.input["x"], b.input["y"])
                new_screenshot = render_calculator(state.display)
                results.append({
                    "type": "tool_result", "tool_use_id": b.id,
                    "content": [
                        {"type": "text", "text": f"Clicked {clicked_label!r}. Display now shows: {state.display}"},
                        {"type": "image", "source": {"type": "base64", "media_type": "image/png",
                                                      "data": image_to_b64(new_screenshot)}},
                    ],
                })
        messages.append({"role": "user", "content": results})
    return state.display, "[budget exhausted]"

final_display, final_text = run_computer_use_agent(
    "Using the on-screen calculator, click 5, then +, then 3, then =, to compute 5 + 3. "
    "Stop once the display shows the final numeric result."
)
print(f"Final calculator display: {final_display!r}")
print(f"Agent's closing message: {final_text!r}")


## Browser agents: a different, usually more robust action space

Browser agents (Playwright/Selenium-driven) CAN act like the pixel-coordinate agent above, but the more robust pattern uses the page's accessibility tree or DOM instead of raw pixels — actions like `click(selector="button#submit")` or `click(role="button", name="Submit")` instead of `click(x=512, y=340)`. This matters because a UI redesign that moves a button's PIXELS breaks a coordinate-based agent instantly, but the button's accessible NAME or role often survives redesigns unchanged — the same robustness argument as preferring semantic HTML/ARIA labels over visual layout in traditional web accessibility work.

In [ ]:
# The two action-space styles side by side, for the SAME click:
pixel_action = {"action": "click", "x": 512, "y": 340}
selector_action = {"action": "click", "selector": "button#submit-order"}
accessibility_action = {"action": "click", "role": "button", "name": "Submit Order"}

print("Pixel-based (computer use):", pixel_action, " -> breaks if the button MOVES")
print("Selector-based (DOM):      ", selector_action, " -> breaks if the id/class is renamed")
print("Accessibility-tree-based:  ", accessibility_action, " -> survives most visual AND markup redesigns")


## Exercises

**Exercise 1 (Warm-up):** Change the goal to `"Compute 9 * 1 using only digit and operator clicks, then stop"` (note: this calculator has no `*` button) and observe how the agent handles a goal it cannot actually achieve with the available tools.

**Exercise 2 (Apply):** Add a `clear` button to `BUTTON_LAYOUT` (and its handling in `CalculatorState.click`) that resets `expression` and `display` to `"0"`. Confirm an agent asked to "compute 5+3, then clear the display" uses it correctly.

**Exercise 3 (Extend):** Notebook 30 covered guardrails for tool-using agents. Sketch what a guardrail would need to check for a computer-use agent specifically — what's the equivalent of "forbidden tool" or "path traversal" when the tool is literally full-screen pixel control?


In [ ]:
# Exercise 1: Warm-up
# Task: Try a goal requiring an operator this calculator doesn't have, observe the agent's behavior.
# Hint: run_computer_use_agent("Compute 9 * 1...") — check final_text for how it reports the gap.

# YOUR CODE HERE


# Exercise 2: Apply
# Task: Add a "C" (clear) button to BUTTON_LAYOUT and CalculatorState.click, then test end-to-end.
# Hint: BUTTON_LAYOUT rows must stay rectangular — replace one existing label or extend the grid.

# YOUR CODE HERE


# Exercise 3: Extend
# Task: Sketch a guardrail for a computer-use agent (full-screen pixel control).
# Hint: think about what regions of the screen should be off-limits regardless of the task
# (e.g. a screen area showing another app's sensitive data) and how you'd enforce that.

# YOUR CODE HERE


<details>
<summary>Click to reveal solutions</summary>

```python
# Exercise 1
display, text = run_computer_use_agent("Compute 9 * 1 using only digit and operator clicks, then stop.")
print(display, text)
# A well-behaved agent should notice there's no '*' button and say so in its final text rather
# than clicking randomly — this is the same "honest about limits" behavior notebook 19's budget
# exercise was checking for, now applied to a UI the agent can visually inspect for capabilities.

# Exercise 2
BUTTON_LAYOUT_V2 = [
    ["7", "8", "9"],
    ["4", "5", "6"],
    ["1", "2", "3"],
    ["0", "+", "C"],   # replaced '=' with 'C'... in practice you'd extend the grid to keep both
]

class CalculatorStateV2(CalculatorState):
    def click(self, x, y):
        for r, row in enumerate(BUTTON_LAYOUT_V2):
            for c, label in enumerate(row):
                x0, y0, x1, y1 = button_bbox(r, c)
                if x0 <= x <= x1 and y0 <= y <= y1:
                    if label == "C":
                        self.expression, self.display = "", "0"
                    elif label == "=":
                        try:
                            self.display = str(eval(self.expression, {"__builtins__": {}}))
                        except Exception:
                            self.display = "Error"
                        self.expression = ""
                    else:
                        self.expression += label
                        self.display = self.expression
                    return label
        return None

# Exercise 3
# A computer-use guardrail needs SCREEN-REGION scoping, not tool-name scoping: an allowlist of
# on-screen bounding boxes the agent may click/type into (e.g. only inside the target app's
# window), a blocklist of regions known to show sensitive content (password managers, other
# users' sessions), and a policy that any click landing outside the allowlist is rejected before
# execution — the pixel-space equivalent of notebook 30's read_file_scoped path allowlist.
```
</details>

## Key Takeaways
- Computer-use and browser agents run the same screenshot → reason → act → repeat loop as any other agent (notebook 19); only the observation (an image) and action (a coordinate or selector) change shape.
- Anthropic's computer-use tools (`computer`, `text_editor`, `bash`) are trained-in action vocabularies, not hand-written JSON schemas like `web_search` — but the surrounding loop is identical to what you already know.
- Accessibility-tree/selector-based browser actions are usually more robust than raw pixel coordinates, because semantic identity (a button's name/role) survives redesigns that break fixed coordinates.
- Reach for computer use only when no API exists for the target software — it's the highest-friction, most brittle tool in the agent toolbox, appropriately a "nice-to-have" rather than a core skill.
- Guardrails for computer-use agents need screen-region scoping (an allowlist of clickable areas), the pixel-space analogue of notebook 30's filesystem path scoping.

## What's Next
Notebook 35 covers voice and realtime pipelines — a different frontier modality with its own latency and interruption-handling concerns.
